In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXX" # Add your namespace-prefix here

### Define Ability

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.find-events.camera-signal",
        description="Identify if the camera signal is abnormal based on changes from its normal environment.",
        worker_release="qwen3-instruct",
        text_prompt="""
          Act as a specialized industrial security analyst. Evaluate the provided frame from a security camera stream and output a binary operational status.

          Analysis Criteria for Failure: Thoroughly check the frame for any of the following low-tech attacks or hardware failures:
          Visual degradation: Diffuse blur (e.g., oil smearing), heavy haze, or loss of high-frequency sharp edges on the lens itself.
          Physical obstruction: The lens surface is directly covered or blinded by opaque materials (e.g., spray paint, tape, cloth).
          Scene invalidity: The camera hardware has been knocked, dropped, or tilted and is now viewing an invalid scene (e.g., pointing directly at the sky, the ground, or a blank wall).
          Signal failure: The frame is completely black or shows a total entropy collapse.

          Crucial Exclusion (False Positives): People, vehicles, or objects moving or standing in the field of view are perfectly normal. Do NOT label the frame as abnormal if a person is standing in front of the camera, even if they take up the majority of the frame. abnormal strictly applies only if the camera hardware itself has been physically altered, repositioned, damaged, or its lens directly coated/covered.

          Categories:
          normal: The camera has clear visibility and its hardware is intact, even if a person or object is currently standing in the frame.
          abnormal: The frame exhibits actual visual degradation, lens obstruction, camera misalignment, or signal failure as defined above.
          Constraint: Output exactly one word: either normal or abnormal. Do not provide any explanations or additional text.
        """,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=150,
            fps=10,
            image_size=512
        ),
        is_public=False
    )
]

### Create Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Video

In [ ]:
from pathlib import Path

pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.find-events.camera-signal:latest",
   )
])

video_path = ... # Add path to video

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   sample_video_path = Path(video_path)
   job = endpoint.upload(sample_video_path)
   while result := job.predict():
      print(json.dumps(result, indent=2))

print("Done")